# Fase 1 — Fundación de datos del Mundial 2026

## 1. Objetivo del proyecto

Este notebook documenta la **fundación de datos** del sistema de pronósticos para el Mundial 2026: qué fuentes usamos, cómo garantizamos su procedencia, y qué contratos estructurales protegen cada artefacto canónico.

El proyecto completo combina modelos estructurales (Elo dinámico, Dixon-Coles) con machine learning (XGBoost) en un ensemble calibrado que alimenta simulaciones Monte Carlo del torneo. Nada de eso es confiable si los datos de entrada no lo son — por eso la Fase 1 existe como fase propia, con gates de aceptación automatizados.

**Contrato pedagógico del repositorio:** toda celda de código va precedida por una celda markdown que explica *qué hacemos y por qué* (`What and why`) y seguida por una celda markdown que interpreta el resultado (`Interpretation`). Este notebook **lee** los artefactos canónicos e **importa** las funciones de producción desde `cdd_mundial.data` — nunca redefine lógica de ingesta, porque esa lógica vive en `src/` donde está cubierta por pruebas.

Los artefactos en `data/processed/` no se versionan: se regeneran desde `data/raw/` + código (ver `README.md`, sección *Reproducibilidad*).

## 2. Configuración y procedencia de las fuentes

**What and why:** Cada captura cruda y cada artefacto derivado tiene un manifiesto de procedencia en `data/metadata/` con su fuente, URL, licencia, versión de snapshot y checksum SHA-256. Esto hace verificable la regla del proyecto de que `data/raw/` es inmutable y que todo lo procesado es regenerable. Aquí fijamos el directorio de trabajo en la raíz del repo (los módulos de `cdd_mundial.data` usan rutas relativas a ella) y leemos todos los manifiestos para inventariar qué tenemos y bajo qué licencia.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
assert (ROOT / 'pyproject.toml').exists(), 'ejecuta desde la raiz del repo o notebooks/'
os.chdir(ROOT)

manifests = []
for path in sorted(Path('data/metadata').glob('*.provenance.json')):
    record = json.loads(path.read_text(encoding='utf-8'))
    manifests.append(
        {
            'artefacto': path.name.replace('.provenance.json', ''),
            'fuente': record['source'],
            'version': record['source_version'],
            'licencia': record['license'][:48],
            'sha256 (prefijo)': record['sha256'][:12],
        }
    )
pd.DataFrame(manifests)

,artefacto,fuente,version,licencia,sha256 (prefijo)
0,en.teams.tsv,eloratings,2026-06-11T18-11-05Z,Source terms not stated; captured for reproducib,ff4b8c9c7bcd
1,fixture_2026.csv,fifa-world-cup-2026-fixture,verified-2026-06-11,Authored normalized reference dataset from publi,61684a33b5de
2,former_names.csv,martj42,2026-06-11,CC0-1.0,2d57f59a3ac1
3,historical_matches.parquet,martj42,2026-06-11,CC0-1.0,067ada821b9e
4,results.csv,martj42,2026-06-11,CC0-1.0,50f17eb331a3
5,shootouts.csv,martj42,2026-06-11,CC0-1.0,e52e503badc1
6,World.tsv,eloratings,2026-06-11T18-11-05Z,Source terms not stated; captured for reproducib,5727bea4a760


**Interpretation:** El inventario muestra las tres familias de fuentes de la Fase 1: el histórico **martj42** de Kaggle (resultados, penales y nombres históricos, licencia CC0-1.0), los TSV de **eloratings.net** (ratings actuales; el sitio no publica términos, por eso la captura se documenta como uso de investigación reproducible) y el **fixture oficial 2026** verificado a mano contra FIFA. Cada checksum permite detectar cualquier mutación posterior del archivo crudo: si un SHA-256 no coincide con su manifiesto, los gates de aceptación fallan. Las cuotas de mercado tienen una política aparte (`data/metadata/odds_provider_policy.json`) porque sus payloads crudos **no** son redistribuibles y viven solo en rutas ignoradas por git.

## 3. Identidades canónicas

**What and why:** El error clásico al combinar fuentes de fútbol es la identidad de equipos: la misma selección aparece como `South Korea`, `Korea Republic` o `KR` según la fuente. Nuestra solución es una tabla maestra de IDs canónicos (slugs revisados a mano, nunca derivados en tiempo de ingesta) más una tabla de alias **exactos** por fuente. La resolución falla ruidosamente (`UnknownTeamError`) ante cualquier nombre no revisado — jamás hay matching difuso silencioso. Verificamos que las 48 selecciones del Mundial resuelven en las cinco fuentes que usa el proyecto.

In [2]:
from cdd_mundial.data.identities import TeamResolver, build_coverage_report

resolver = TeamResolver.from_csv()
participants = resolver.teams.loc[resolver.teams['is_world_cup_2026'], 'team_id']
print(f'Selecciones del Mundial 2026 en la tabla maestra: {participants.size}')

sources = ['martj42', 'eloratings', 'fifa', 'fixture', 'odds']
coverage = build_coverage_report(participants, sources, resolver)
coverage.groupby('source')['resolved'].agg(resueltas='sum', requeridas='count')

Selecciones del Mundial 2026 en la tabla maestra: 48


,resueltas,requeridas
source,,
eloratings,48,48
fifa,48,48
fixture,48,48
martj42,48,48
odds,48,48


**Interpretation:** Las 48 selecciones resuelven en las cinco fuentes (48/48 por fila). Esto significa que cualquier dato nuevo de esas fuentes se enlaza a un ID canónico estable sin ambigüedad. Detalle importante: una fuente puede tener **varios** alias revisados para el mismo equipo (el proveedor de cuotas envía `Bosnia & Herzegovina` además de `Bosnia and Herzegovina`); eso es cobertura legítima, no duplicación, porque la resolución se indexa por el nombre exacto de la fuente y los duplicados verdaderos los rechaza el esquema de alias.

## 4. Semántica de martj42: marcadores y penales

**What and why:** El dataset martj42 registra los marcadores **incluyendo tiempo extra** pero **excluyendo los goles de la tanda de penales**; el ganador por penales vive en un archivo aparte. Si esto se ignora, un modelo de goles aprendería de marcadores contaminados o perdería los desenlaces reales de eliminatorias. Nuestro pipeline preserva esa semántica: los marcadores no se tocan y el ganador por penales se guarda en la columna separada `shootout_winner_team_id`. Lo comprobamos con el caso más famoso: la final del Mundial 2022 (Argentina 3-3 Francia, penales para Argentina).

In [3]:
from cdd_mundial.data.contracts import HistoricalMatchesSchema

historical = HistoricalMatchesSchema.validate(
    pd.read_parquet('data/processed/historical_matches.parquet')
)
print(f'Partidos completados: {len(historical):,}')
print(f"Rango de fechas: {historical['date'].min()} -> {historical['date'].max()}")
print(f"Definidos en penales: {historical['shootout_winner_team_id'].notna().sum():,}")

columnas = [
    'date', 'home_team_id', 'away_team_id', 'home_score', 'away_score',
    'result_after_extra_time', 'shootout_winner_team_id',
]
historical.loc[
    (historical['date'] == '2022-12-18') & (historical['tournament'] == 'FIFA World Cup'),
    columnas,
]

Partidos completados: 49,405
Rango de fechas: 1872-11-30 -> 2026-06-10
Definidos en penales: 677


,date,home_team_id,away_team_id,home_score,away_score,result_after_extra_time,shootout_winner_team_id
45786,2022-12-18,argentina,france,3,3,True,argentina


**Interpretation:** La final de 2022 aparece como `3-3` — exactamente lo que el marcador tras 120 minutos fue — con `shootout_winner_team_id = argentina` en su columna separada. El esquema estricto (`HistoricalMatchesSchema`, pandera con `strict=True`) garantiza además que cada fila tiene IDs canónicos resueltos, marcadores no negativos y equipos distintos. La bandera `result_after_extra_time` se deriva de la presencia de tanda de penales en la fuente: marca los partidos cuyo marcador final llegó tras la prórroga completa. Con ~49 mil partidos desde 1872 tenemos historia de sobra para recomputar Elo propio en la Fase 2.

## 5. Auditoría de sedes neutrales

**What and why:** La ventaja de localía es uno de los efectos más fuertes en fútbol internacional, y la bandera `neutral` de martj42 es la única forma de saber si el equipo *home* jugaba realmente en casa. En un Mundial casi todos los partidos son en sede neutral — excepto para los anfitriones. Auditamos qué proporción del histórico es neutral, separando los partidos de Mundial del resto, porque esa distribución decide cómo se modela la localía (en Elo: +100 puntos solo cuando no es neutral).

In [4]:
audit = (
    historical.assign(
        contexto=historical['tournament'].where(
            historical['tournament'] == 'FIFA World Cup', 'Otros torneos'
        )
    )
    .groupby('contexto')
    .agg(partidos=('neutral', 'size'), proporcion_neutral=('neutral', 'mean'))
    .round(3)
)
audit

,partidos,proporcion_neutral
contexto,,
FIFA World Cup,964,0.874
Otros torneos,48441,0.252


**Interpretation:** Los partidos de Mundial son mayoritariamente neutrales — la fracción no neutral corresponde a los anfitriones de cada edición, que sí juegan en casa. Fuera de Mundiales la mayoría de los partidos tiene local verdadero. Consecuencia para el modelado: en 2026 los partidos de México, Estados Unidos y Canadá en su territorio deben tratarse como localía real, y el resto del torneo como neutral. Ignorar esta distinción sesgaría sistemáticamente las probabilidades a favor del equipo listado como `home` en el fixture.

## 6. Cobertura Elo actual

**What and why:** El rating Elo es el insumo estructural central del baseline. Aquí validamos el **snapshot actual** de eloratings.net (capturado con procedencia y resuelto a IDs canónicos): debe cubrir exactamente a las 48 selecciones participantes, sin huecos, porque un solo equipo sin rating rompería la simulación del torneo. La **recomputación de Elo propio** desde los 49 mil partidos históricos (formula World Football Elo: K por torneo, multiplicador por margen de gol, localía) es entregable de la Fase 2; este snapshot funciona como referencia y cross-check.

In [5]:
from cdd_mundial.data.contracts import EloRatingsSchema

elo = EloRatingsSchema.validate(pd.read_parquet('data/processed/elo_current.parquet'))
print(f"Cobertura: {elo['team_id'].nunique()} de 48 selecciones")
print(f"Snapshot: {elo['rating_date_utc'].iloc[0]} (fuente: {elo['source'].iloc[0]})")

elo.sort_values('elo_rating', ascending=False).head(10)[
    ['team_id', 'elo_rating', 'rank']
].reset_index(drop=True)

Cobertura: 48 de 48 selecciones
Snapshot: 2026-06-11T18:11:05.549548Z (fuente: eloratings)


,team_id,elo_rating,rank
0,spain,2157.0,1
1,argentina,2115.0,2
2,france,2063.0,3
3,england,2024.0,4
4,brazil,1991.0,5
5,portugal,1989.0,6
6,colombia,1982.0,7
7,netherlands,1948.0,8
8,ecuador,1938.0,9
9,germany,1932.0,10


**Interpretation:** Cobertura 48/48 — ninguna selección llega al simulador sin rating. El top 10 es plausible futbolísticamente (potencias europeas y sudamericanas encabezan), lo que sirve de validación de cara: si aquí apareciera un equipo menor arriba, sospecharíamos un error de mapeo de alias. La columna `rank` es el ranking global de eloratings.net, así que valores mayores a 48 son normales: hay más de 200 selecciones en el mundo y no todas clasificaron.

## 7. Estructura del fixture 2026

**What and why:** El fixture congelado es el esqueleto de la simulación: 104 partidos (72 de grupos en 12 grupos de 4, más 32 de eliminatoria directa desde la nueva Ronda de 32). Lo cargamos con `load_fixture_2026`, que no solo lee el CSV revisado a mano contra FIFA sino que **valida el contrato completo del torneo**: conteos por fase y por grupo, cada equipo exactamente 3 veces en grupos, sin emparejamientos duplicados, y referencias de slots (`1A`, `W73`...) para los cruces aún no determinados.

In [6]:
from cdd_mundial.data.ingest_fixture import load_fixture_2026

fixture = load_fixture_2026()
print(f'Partidos totales: {len(fixture)}')
print(f"Fase de grupos: {(fixture['stage'] == 'group').sum()}")
print(f"Grupos: {sorted(fixture['group'].dropna().unique())}")

fixture.groupby('stage').size().sort_values(ascending=False).rename('partidos').to_frame()

Partidos totales: 104
Fase de grupos: 72
Grupos: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']


,partidos
stage,
group,72
round_of_32,16
round_of_16,8
quarterfinal,4
semifinal,2
final,1
third_place,1


**Interpretation:** 104 partidos, 72 de grupos, grupos A-L con 6 partidos cada uno y el árbol completo de eliminatorias (32 + 16 + 8 + 4 + 2 de semifinal, tercer puesto y final). Los partidos de eliminatoria conservan sus *slots* oficiales en vez de adivinar participantes: el simulador de la Fase 3 los resolverá dinámicamente con los resultados (reales o simulados) de la fase previa. Como el validador corre en cada carga, cualquier edición accidental del CSV congelado rompe las pruebas de inmediato.

## 8. Cuotas de mercado y de-margining

**What and why:** Las cuotas de las casas de apuestas son el benchmark externo contra el que mediremos nuestros pronósticos — el mercado agrega información que ningún modelo individual tiene. Pero las probabilidades implícitas (`1/precio`) suman más de 1 porque incluyen el margen de la casa (*overround*). `demargin_decimal_odds` lo elimina con normalización multiplicativa. Mostramos el mecanismo con un ejemplo y luego resumimos el benchmark materializado. **Política de datos:** los payloads crudos del proveedor no son redistribuibles y viven solo bajo `data/raw/odds/` (ignorado por git); aquí solo se muestran agregados de-margined, que sí son publicables. Esto no es asesoría de apuestas.

In [7]:
from cdd_mundial.data.contracts import OddsSchema
from cdd_mundial.data.ingest_odds import demargin_decimal_odds

ejemplo = [1.95, 3.4, 4.4]
crudas = [round(1 / p, 4) for p in ejemplo]
print(f'Precios decimales (1/X/2): {ejemplo}')
print(f'Prob. implicitas crudas:   {crudas}  (suma = {sum(crudas):.4f})')
print(f'Prob. de-margined:         {[round(p, 4) for p in demargin_decimal_odds(ejemplo)]}')

odds = OddsSchema.validate(pd.read_parquet('data/processed/odds_2026.parquet'))
print(f"\nBenchmark: {len(odds):,} filas | {odds['match_id'].nunique()} partidos | "
      f"{odds['bookmaker'].nunique()} casas | overround mediano "
      f"{odds['overround'].median():.4f}")

consenso = (
    odds.groupby(['match_id', 'home_team_id', 'away_team_id'])[
        ['prob_home', 'prob_draw', 'prob_away']
    ]
    .median()
    .round(4)
    .head(6)
)
consenso

Precios decimales (1/X/2): [1.95, 3.4, 4.4]


Prob. implicitas crudas:   [0.5128, 0.2941, 0.2273]  (suma = 1.0342)
Prob. de-margined:         [0.4959, 0.2844, 0.2198]

Benchmark: 1,385 filas | 71 partidos | 24 casas | overround mediano 1.0521


,,,prob_home,prob_draw,prob_away
match_id,home_team_id,away_team_id,,,
WC26-002,south-korea,czechia,0.3671,0.3130,0.3202
WC26-003,canada,bosnia-and-herzegovina,0.5191,0.2709,0.2106
WC26-004,united-states,paraguay,0.4943,0.2790,0.2276
WC26-005,haiti,scotland,0.1614,0.2252,0.6137
WC26-006,australia,turkiye,0.1822,0.2530,0.5653
WC26-007,brazil,morocco,0.5746,0.2533,0.1741


**Interpretation:** El ejemplo muestra el efecto del margen: las probabilidades crudas suman ~1.04 (un 4% de margen) y tras el de-margining suman exactamente 1. En el benchmark real cada fila es una cotización (partido x casa) ya orientada al local/visitante del fixture canónico, y el esquema `OddsSchema` rechaza cualquier fila cuyas probabilidades normalizadas no sumen 1. El **consenso mediano** por partido es la referencia que usaremos en la evaluación: si nuestro ensemble no acerca su log-loss al del mercado, sabremos cuánto nos falta. Los favoritos del consenso coinciden con la intuición futbolística, otra validación de cara.

## 9. Conclusiones

La fundación de datos de la Fase 1 queda establecida y verificable:

1. **Procedencia** — todo artefacto tiene manifiesto con fuente, licencia, versión y SHA-256; `data/raw/` es inmutable y `data/processed/` se regenera desde raw + código.
2. **Identidades** — 48 selecciones canónicas con resolución exacta (sin fuzzy matching) en martj42, eloratings, FIFA, fixture y cuotas.
3. **Histórico** — ~49 mil partidos (1872-2026) con la semántica de martj42 preservada: marcadores con tiempo extra, penales en columna separada.
4. **Elo actual** — snapshot con cobertura 48/48 como referencia; el Elo propio recomputado llega en la Fase 2.
5. **Fixture** — 104 partidos congelados y validados estructuralmente contra el formato FIFA 2026.
6. **Mercado** — benchmark de-margined de cuotas tres-vías enlazado al fixture, con los payloads crudos fuera del repositorio público por sus términos de uso.

Todo lo anterior está cubierto por gates automatizados (`python -m pytest -q tests/test_phase1_acceptance.py`). Las siguientes fases construyen sobre estos contratos: features y Elo dinámico (Fase 2), Dixon-Coles y simulador Monte Carlo (Fase 3), pipeline de actualización por jornada (Fase 4).